
In the problem of sparse approximation, we are trying to approximate a given vector $y$ as a linear combination of few columns of a given matrix $A$.
It is useful in many applications, such as machine learning (e.g. feature selection), image processing (e.g. image denoising, inpainting, deblurring, compression), and signal processing (e.g. compressive sensing).

In this post, we will discuss greedy algorithms for solving the sparse approximation problem, focusing on the well-known Matching Pursuit (MP) and Orthogonal Matching Pursuit (OMP) algorithms, as well as a "new" algorithm that I call the Most Obvious Matching Pursuit (MOMP).

# Notation
We will consider the following flavor of the sparse approximation problem:
\begin{equation}
    \begin{array}{ll}
        \underset{x}{\mbox{minimize}}  & \| Ax - y\|^2 \\
        \mbox{subject to} & \text{$x$ has at most $k$ non-zero entries}.
    \end{array}
    \label{e-opt-prob}
\end{equation}
Where

* $y$ is the $m$-vector we are trying to approximate,

* $A$ is the $m \times n$ dictionary matrix with columns $a_1, \ldots, a_n$,

* $x$ is the $n$-vector of coefficients,

When $S$ is an ordered subset of $1, \ldots, n$,
we denote by $A_S$ the matrix obtained by keeping only the columns of the matrix $A$ that are in $S$, and by $x_S$ the vector obtained by keeping only the elements of $x$ that are in $S$.

For a matrix $M$, $\text{Range}(M)$ denotes it's column space, and $M^H$ denotes it's conjugate transpose (if you don't care about complex data, you can think of it as the transpose).

# Support Recovery
If the support of $x$ is known and denoted by $S$, then since $A x = A_S x_S$, finding the optimal $x_S$ is a simple least squares problem:
$$
\underset{x_S}{\text{minimize}}  \| A_S x_S - y \|^2.
$$
This means that in practice, the problem is to recover the support of $x$.
The following results will be useful in this context:

* An optimal support $S$ maximizes $y^H P_{A_S} y$,
where $P_{A_S}$ is the projection matrix onto $\text{Range}(A_S)$.
<!-- (The intution here is that with an optimal support, $y$ should be approximately in $\text{Range}(A_s)$, so projecting onto it should have little effect). -->

* An optimal $x_S$ is given by any solution to $A_S^H A_S x_S = A_S^H y$ (aka the normal equations).

::: {.callout-note icon=false collapse=true}
#### Proof [^1]
[^1]: While these are standard results for least squares problems,
I find that they are not always presented in a clear way.
I like proving such results without using gradients
(which obscure the intuition, in my opinion)
and without inverting matrices (which adds unnecessary caveats about rank and conditioning).

First, we rewrite $y$ as $y^{||} + y^{\perp}$, where $y^{||}$ is in $\text{Range}(A_S)$ and $y^{\perp}$ is orthogonal to it.
Since $A_S x_S$ is in $\text{Range}(A_S)$, we can apply the Pythagorean theorem (twice) to get:
\begin{align}
 \| A_S x_S - y \|^2 &=
\| \left(A_S x_S - y^{||}\right) - y^{\perp} \|^2
 \\&=
  \| A_S x_S - y^{||} \|^2 + \| y^{\perp} \|^2 
  \\&=
  \| A_S x_S - y^{||} \|^2 + \| y \|^2 - \| y^{||} \|^2 .
\end{align}
Minimizing across $x_S$, the first term vanishes (since $y^{||}$ is in $\text{Range}(A_S)$, there is by definition an $x_S$ such that $y^{||} = A_S x_S$).
\
Thus, an optimal $S$ maximizes $\| y^{||} \|^2$.
\
Since $P_{A_S}$ is a projection matrix, it is symmetric and idempotent, so
$$
\| y^{||} \|^2 = \| P_{A_S} y \|^2 = y^H P_{A_S}^H P_{A_S} y = y^H P_{A_S} y
$$
which proves the first item.

An optimal $x_S$ satisfies $y^{||} = A_S x_S$.
\
As the projection of $y$ onto $\text{Range}(A_S)$, $y^{||}$ is characterized by $A_S ^H \left(y - y^{||} \right) = 0$.
\
Combining the two, we get the second item.
:::

# Greedy Algorithms
Exactly Solving the sparse approximation problem is NP-hard. It turns out there isn't a significantly better way than brute force checking all $n \choose k$ possible supports.
Thus we often turn to greedy algorithms[^convex], which are faster but not guaranteed to find the optimal solution.
We will present 3 such algorithms and their Python implementations.
We will rely on the following helper functions:

In [1]:
import numpy as np


def project_y_onto_range_A_S(A: np.ndarray, y: np.ndarray, S: list[int]) -> np.ndarray:
    """
    Returns $P_{A_S} y$
    """
    A_S = A[:, S]
    Q, _ = np.linalg.qr(A_S)
    return Q @ Q.H @ y


def score_S(A: np.ndarray, y: np.ndarray, S: list[int]) -> float:
    """
    Returns $y^H P_{A_S} y$
    """
    A_S = A[:, S]
    Q, _ = np.linalg.qr(A_S)
    z = Q.H @ y
    return np.real(z.H @ z)  # real is needed only because of numerical errors

::: {.callout-note icon=false collapse=true}
#### Why QR?
You may wonder why we are using the QR decomposition to compute the projection, and not simply solving the normal equations directly to get $x_S$ and then computing $A_S x_S$ to get the projection.
The reason is that the normal equations are not always well-conditioned, which may lead to numerical errors in $x_S$, which in turn may lead to numerical errors in the projection.
But we don't really care about $x_S$, we only care about the projection.
The QR decomposition is used to compute the projection directly in a more numerically stable way, without the need to compute the intermediate $x_S$.
:::

[^convex]: Convex relaxation (e.g. Basis Pursuit, LASSO) is another approach, but it is not the focus of this post.

## Matching Pursuit (MP)
MP is a simple and popular iterative algorithm for the sparse approximation problem.
We start with an empty support. Then, at each iteration:

1. Add the column $i$ that maximizes $y^H P_{a_i} y$ to the support.

2. Update $y$ by projecting it onto the space orthogonal to $a_i$.

In Python, it looks like this:

In [3]:
def mp(A: np.ndarray, y: np.ndarray, k: int) -> set[int]:
    S = set()
    n = A.shape[1]
    while len(S) < k:
        i = max(range(n), key=lambda i: score_S(A, y, [i]))
        S.add(i)
        y -= project_y_onto_range_A_S(A, y, [i])

::: {.callout-note icon=false collapse=true}
#### A more efficient implementation
In MP, we are always considering a support of size 1, so the normal equations are actually scalar equations:
\begin{align*}
a_i^H a_i x_i = a_i^H y
\end{align*}
If we normalize the columns of $A$ to have unit norm, we get very simple equations for $x_i$ and $y^H P_{a_i} y$:
\begin{align*}
x_i &= a_i^H y, 
\\ &y^H P_{a_i} y &= | x_i |^2.
\end{align*}
In Python, this can be implemented as follows:
```python
def mp(A: np.ndarray, y: np.ndarray, k: int) -> set[int]:
    S = set()
    A = A / np.linalg.norm(A, axis=0)
    while len(s) < k:
        x = A.H @ y
        i = np.argmax(np.abs(x))
        S.add(i)
        y -= A[:, i] * x[i]
```
:::

## Orthogonal Matching Pursuit (OMP)
OMP is a popular variant of MP.
While in MP we project $y$ onto the space orthogonal to the selected column in the current iteration, in OMP we project $y$ onto the space orthogonal to all the selected columns so far:

In [ ]:
def mp(A: np.ndarray, y: np.ndarray, k: int) -> set[int]:
    S = set()
    n = A.shape[1]
    while len(S) < k:
        i = max(range(n), key=lambda i: score_S(A, y, [i]))
        S.add(i)
        y -= project_y_onto_range_A_S(A, y, list(S))

::: {.callout-note icon=false collapse=true}
#### A more efficient implementation
Like in MP, if the columns are normalized, the selected column is the one with the largest absolute inner product with $y$.
Unlike MP, we can't reuse the inner product to compute the projection of $y$ onto the space orthogonal to the selected columns.
However, there are ways to speed up the computation of the projection by using the solution of the previous iteration, using e.g. incremental QR factorizations.
I plan to discuss this in a future post.
:::

## The Most Obvious Matching Pursuit (MOMP)
Start with $S=\emptyset$. At each iteration add the column $i$ that maximizes $y^H P_{A_{S \cup \left\{ i \right\}}} y$:

In [ ]:
def momp(A: np.ndarray, y: np.ndarray, k: int) -> set[int]:
    S = set()
    n = A.shape[1]
    while len(S) < k:
        S.add(max(range(n), key=lambda i: score_S(A, y, list(S) + [i])))
    return S

That's it, there are no $y$ updates here.

OMP tries to improve MP by optimizing the coefficients of all the selected columns at each iteration.
However, the coefficients optimization happens only after the new column was already added. During the selection of the new column, like MP, OMP selects the optimal column assuming $k=1$, affectively optimizing only the coefficient of the selected column.

In MOMP, we fix that, by simultaneously optimizing both the new column and the coefficients of all the selected columns. That is, we are affectively solving 
$$
\underset{S, x_S}{\text{minimize}} 
 \|
    A_S x_S - y 
\|^2 
$$
at each iteration, with the constraint that $S$ is the same as in the previous iteration, except for one additional column.

MP and OMP are well known and widely used.
However, I have not seen the algorithm MOMP in the literature (I would be happy to be proven wrong), even though I really think it is the most obvious greedy algorithm to try.

A possible reason for this is it's compuation complexity: at each iteration, MOMP requires solving a least squares problem for each candidate column, as opposed to OMP which requires solving it only for the new column.
\
There are however ways to utilize the solution of the previous iteration to speed up the computation (by using incremental Cholesky/QR factorizations). I plan to discuss this in a future post.